# Testing Notebook for setup

In [3]:
import torch

print(torch.backends.mps.is_available())

True


In [4]:
from datasets import load_dataset

dataset = load_dataset("CarperAI/openai_summarize_tldr")

README.md:   0%|          | 0.00/532 [00:00<?, ?B/s]

data/train-00000-of-00001-e8c59e5cf7bce1(…):   0%|          | 0.00/111M [00:00<?, ?B/s]

data/test-00000-of-00001-59ffb27399371ea(…):   0%|          | 0.00/6.23M [00:00<?, ?B/s]

data/valid-00000-of-00001-0e33e6bd86e3ed(…):   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/116722 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6553 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/6447 [00:00<?, ? examples/s]

In [25]:
dataset_length = 0
for i in dataset.keys():
    dataset_length += len(dataset[i])

print(f'training examples: {100*len(dataset['train'])/dataset_length}%')
print(f'validation examples: {100*len(dataset['valid'])/dataset_length}%')
print(f'test examples: {100*len(dataset['test'])/dataset_length}%')

training examples: 89.97856955643607%
validation examples: 4.969858620742819%
test examples: 5.05157182282111%


Dataset consists of a total of around -130k examples with a split of around roughly 90% train 5% validation and 5% training, example of a training row can be seen below, the iterator within the dataset train has two keys `prompt` and `label`

In [12]:
print(dataset["train"][0]['prompt'])
print("-"*50)
print(dataset["train"][0]['label'])

SUBREDDIT: r/relationships
TITLE: I (f/22) have to figure out if I want to still know these girls or not and would hate to sound insulting
POST: Not sure if this belongs here but it's worth a try. 

Backstory:
When I (f/22) went through my first real breakup 2 years ago because he needed space after a year of dating roand  it effected me more than I thought. It was a horrible time in my life due to living with my mother and finally having the chance to cut her out of my life. I can admit because of it was an emotional wreck and this guy was stable and didn't know how to deal with me. We ended by him avoiding for a month or so after going to a festival with my friends. When I think back I wish he just ended. So after he ended it added my depression I suffered but my friends helped me through it and I got rid of everything from him along with cutting contact. 

Now: Its been almost 3 years now and I've gotten better after counselling and mild anti depressants. My mother has been out of m

# Create Training Examples

Since we want our LLM to predict on next sequence, we can combine the labels and the prompts for training purposes. Usually we would need to create a token and place it inbetween the prompt and labels to signal that the summary has begun, however since the prompt always ends with 'TLDR' we can use this as a token provided we extend the existing gpt2 vocab to handle this special token. We do though infact need to end the sentence with a special signal to tell the LLM that this is the end, here we use `<|endoftext|>`

In [88]:
import pandas as pd
train = pd.DataFrame(columns=['text'])
test = pd.DataFrame(columns=['text'])
val = pd.DataFrame(columns=['text'])

combined_text = [] 
for i in range(len(dataset['train'])):
    combined_text.append(dataset['train'][i]['prompt'] + " " + dataset['train'][i]['label'])
train['text'] = combined_text

combined_text_test = []
for i in range(len(dataset['test'])):
    combined_text_test.append(dataset['test'][i]['prompt'] + " " + dataset['test'][i]['label'])
test['text'] = combined_text_test


combined_text_val = []
for i in range(len(dataset['valid'])):
    combined_text_val.append(dataset['valid'][i]['prompt'] + " " + dataset['valid'][i]['label']) 
val['text'] = combined_text_val

# Tokenise the text (GPT-2)

In [98]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
tensors = tokenizer(train['text'].tolist(), return_tensors='pt', padding="max_length", max_length=512, truncation=True)

Here we print the attention mask which we need for the training process, note that for the padded tokens (<|endoftext|>) we have a value of 0 within the attention mask

In [114]:
tensors['attention_mask'][0]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

For language model training the labels are just the input_ids shifted by one position (predict next token). In practice HuggingFace handles the shift internally:

In [111]:
labels = tensors['input_ids'].clone()

We want to set the values of the padded tokens within the labels to be -100, this is because the Loss function is specifically coded to ignore this specific value.

In [122]:
bool_mask = (labels == 50256)
labels[bool_mask] = -100
labels

tensor([[   50, 10526, 22083,  ...,  -100,  -100,  -100],
        [   50, 10526, 22083,  ...,  -100,  -100,  -100],
        [   50, 10526, 22083,  ...,  -100,  -100,  -100],
        ...,
        [   50, 10526, 22083,  ...,  -100,  -100,  -100],
        [   50, 10526, 22083,  ...,  -100,  -100,  -100],
        [   50, 10526, 22083,  ...,  -100,  -100,  -100]])

In [129]:
from torch.utils.data import Dataset, random_split

class CustomDataset(Dataset):
    def __init__(self, input_ids, labels, attention_mask, transform=None, target_transform=None):
        self.input_ids = input_ids
        self.labels = labels
        self.attention_mask = attention_mask

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.labels[idx], self.attention_mask[idx]

In [130]:
data = CustomDataset(tensors['input_ids'], labels, tensors['attention_mask'])

train, test, val = random_split(data, [0.9,0.05,0.05])

In [144]:
from torch.utils.data import DataLoader
train_dataloader = DataLoader(torch.utils.data.Subset(train, range(10000)), batch_size=16, shuffle=True)
val_dataloader = DataLoader(val, batch_size=16, shuffle=False)
test_dataloader = DataLoader(test, batch_size=16, shuffle=False)

In [141]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("gpt2").to('mps')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [147]:
from torch.optim import AdamW

opt = AdamW(model.parameters(), lr=0.001)
torch.mps.empty_cache()
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels, attention_mask in train_dataloader:
        #Zero the gradients
        opt.zero_grad()

        #forward pass
        outputs = model(input_ids=inputs.to('mps'), attention_mask=attention_mask.to('mps'), labels=labels.to('mps'))
        loss = outputs.loss

        loss.backward()

        opt.step()

        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_dataloader)}")

RuntimeError: MPS backend out of memory (MPS allocated: 20.34 GiB, other allocations: 27.81 MiB, max allowed: 20.40 GiB). Tried to allocate 72.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).